>  이 파일은 해설 버전입니다.

# 3. 검색 기초 - 키워드 검색 (Keyword Search)

## 학습 목표
- 역색인(Inverted Index) 구조를 이해한다
- Full Text Query (Match)와 Exact Query (Term)의 차이를 학습한다
- 복합 쿼리 (Bool Query)와 필터링을 실습한다

## 3.1 OpenSearch 접속 설정
### OpenSearch 실습 환경 접속 가이드

1. 필요 패키지 설치

```bash
pip install opensearch-py openai
```

2. OpenSearch 서버 접속 설정

```python
from opensearchpy import OpenSearch

# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"
OPENSEARCH_PORT = 443
OPENSEARCH_USER = "sdsrag"
OPENSEARCH_PASS = "Student.Rag1!"

# 클라이언트 생성
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_show_warn=False,
)

# 접속 확인
print(client.info()["version"]["number"])
```

3. 인덱스 이름 규칙

```python
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"
# 예: student_01_company_docs, student_02_company_docs, ...
```

4. 접속 확인 테스트

```python
# 클러스터 상태 확인
print(client.cluster.health()["status"])

# 내 인덱스 목록 확인
print(client.cat.indices(index="student_*", format="json"))
```

5. OpenSearch Dashboards (웹 UI)

- 주소: https://sdsos.baeum.ai.kr/dashboard
- 계정: sdsrag / Student.Rag1!
- 로그인 후 좌측 메뉴 → Discover → student_* 인덱스 패턴 선택
- 노트북에서 인덱싱한 문서를 검색/시각화할 수 있음

6. 주의사항

- STUDENT_NUMBER를 반드시 본인 번호로 변경할 것
- 다른 학생의 인덱스(student_XX_*)에는 접근할 수 없음
- OpenAI API 키는 각자 본인 키를 사용할 것

In [ ]:
# [03-01] 필요 패키지 설치
# [목적] OpenSearch 연결과 OpenAI API에 필요한 라이브러리를 설치합니다
# [주의] 처음 한 번만 실행합니다. 오류 시 런타임 다시 시작 후 재실행하세요
# 필요 패키지 설치
!pip install -q opensearch-py openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 7.6 MB/s eta 0:00:00


In [ ]:
# [03-02] 라이브러리 임포트
# [목적] OpenSearch 클라이언트 모듈을 불러옵니다
from opensearchpy import OpenSearch

In [ ]:
# [03-03] OpenSearch 접속 정보 설정
# [목적] 실습용 OpenSearch 서버의 주소, 계정, 포트를 변수로 설정합니다
# [주의] STUDENT_NUMBER를 반드시 본인 번호(0~30)로 변경하세요!
# ★ 본인 번호로 변경 (0~30) ★
STUDENT_NUMBER = 0

# 서버 접속 정보
OPENSEARCH_HOST = "sdsos.baeum.ai.kr"
OPENSEARCH_PORT = 443
OPENSEARCH_USER = "sdsrag"
OPENSEARCH_PASS = "Student.Rag1!"

In [ ]:
# [03-04] OpenSearch 연결 확인
# [목적] 설정한 정보로 OpenSearch 서버에 연결하고 버전을 확인합니다
# [주의] 여기서 오류 나면 이후 모든 셀이 실패합니다. 접속 정보를 재확인하세요
# 클라이언트 생성
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=True,
    verify_certs=True,
    ssl_show_warn=False,
)

# 접속 확인
print(client.info()["version"]["number"])

2.19.4


In [ ]:
# [03-05] 인덱스 이름 설정
# [목적] 본인 번호에 맞는 전용 인덱스 이름을 설정합니다
# [개념] 2장에서 생성한 인덱스를 그대로 사용합니다. 다른 번호 인덱스는 접근 불가
# 본인 번호에 맞는 인덱스 사용 (다른 번호 인덱스 사용 금지)
INDEX_NAME = f"student_{STUDENT_NUMBER:02d}_company_docs"
# 예: student_01_company_docs, student_02_company_docs, ...

In [ ]:
# [03-06] 클러스터 상태 확인
# [목적] OpenSearch 클러스터 상태와 인덱스 목록을 확인합니다
# [주의] 2장의 인덱싱을 먼저 완료해야 검색할 문서가 존재합니다
# 클러스터 상태 확인
print(client.cluster.health()["status"])

# 내 인덱스 목록 확인
print(client.cat.indices(index="student_*", format="json"))

print(f"사용 인덱스: {INDEX_NAME}")

yellow
[{'health': 'green', 'status': 'open', 'index': 'student_00_company_docs', 'uuid': 'IfO-8LeqT_aDsW1AzLtIYQ', 'pri': '1', 'rep': '0', 'docs.count': '20', 'docs.deleted': '0', 'store.size': '775.6kb', 'pri.store.size': '775.6kb'}]
사용 인덱스: student_00_company_docs


In [ ]:
# [03-07] search_and_print 함수 정의
# [목적] 검색 쿼리를 실행하고 결과를 보기 좋게 출력하는 헬퍼 함수입니다
# [개념] 검색 결과에서 점수(_score), 제목, 카테고리, 하이라이트를 추출하여 표시합니다
def search_and_print(query: dict, description: str = ""):
    """검색 실행 및 결과 출력"""
    print(f"=== {description} ===")
    response = client.search(index=INDEX_NAME, body=query)  # OpenSearch 검색 API 호출
    total = response['hits']['total']['value']  # 총 매칭 문서 수
    print(f"검색 결과: {total}건\n")

    for hit in response['hits']['hits']:  # 검색 결과 문서 순회
        src = hit['_source']  # 원본 문서 데이터
        score = hit['_score']  # BM25 관련성 점수
        print(f"[{score:.4f}] {src['title']}")
        print(f"         카테고리: {src['category']}")
        # 하이라이트가 있으면 출력
        if 'highlight' in hit:  # 매칭된 부분을 <em> 태그로 표시
            for field, highlights in hit['highlight'].items():
                print(f"         매칭: {highlights[0][:100]}...")
        print()

    return response


## 3.2 역색인(Inverted Index) 이해

### 역색인이란?
단어 → 문서 매핑 구조 (일반 색인의 반대)

### 텍스트 분석 과정

In [ ]:
# [03-08] Analyze API로 토큰화 확인
# [목적] OpenSearch가 텍스트를 어떻게 토큰(단어 단위)으로 분해하는지 확인합니다
# [개념] 역색인의 핵심: 문장 → 개별 토큰으로 분해 → 각 토큰이 어느 문서에 있는지 매핑
# Analyze API로 토큰화 과정 확인
response = client.indices.analyze(
    index=INDEX_NAME,
    body={
        "analyzer": "standard",  # 기본 분석기: 공백+구두점 기준 토큰 분리
        "text": "회사 연차 휴가 정책 안내"  # 분석할 원문 텍스트
    }
)

print("토큰화 결과:")
for token in response['tokens']:  # 분리된 토큰 목록 순회
    print(f"  [{token['position']}] '{token['token']}' (offset: {token['start_offset']}-{token['end_offset']})")  # position=순서, offset=원문 위치


토큰화 결과:
  [0] '회사' (offset: 0-2)
  [1] '연차' (offset: 3-5)
  [2] '휴가' (offset: 6-8)
  [3] '정책' (offset: 9-11)
  [4] '안내' (offset: 12-14)


## 3.3 Full Text Query (Match)
### Match Query
- 검색어를 **분석(토큰화)** 후 검색
- 전문 검색에 사용
- 기본적으로 OR 조건

In [ ]:
# [03-09] Match Query - 기본 키워드 검색
# [목적] Match 쿼리로 content 필드에서 "VPN 설정"을 검색합니다
# [개념] Match = 검색어를 토큰화한 뒤 OR 조건으로 매칭. 키워드 검색의 기본 방식입니다
# 기본 Match Query - 키워드 검색의 강점 (정확한 기술 용어)
query = {
    "query": {
        "match": {  # Match: 검색어를 토큰화 후 매칭
            "content": "VPN 설정"  # "VPN", "설정" 각각 토큰화 → OR 조건 매칭
        }
    },
    "highlight": {  # 매칭된 부분을 하이라이트 처리
        "fields": {"content": {}}  # content 필드의 매칭 부분 표시
    }
}

search_and_print(query, "Match Query - 'VPN 설정' (정확한 기술 용어)")


=== Match Query - 'VPN 설정' (정확한 기술 용어) ===
검색 결과: 0건



{'took': 2,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [ ]:
# [03-10] Match Query (AND) - 모든 단어 필수
# [목적] AND 연산자로 모든 검색어가 포함된 문서만 찾습니다
# [개념] operator:"and" = 모든 단어가 반드시 포함되어야 매칭. 기본값은 "or"입니다
# Match Query with AND operator - 도구명 검색 (키워드 강점)
query = {
    "query": {
        "match": {
            "content": {
                "query": "SonarQube 정적 분석",
                "operator": "and"  # 모든 토큰이 반드시 포함 (기본값: "or")
            }
        }
    }
}

search_and_print(query, "Match Query (AND) - 'SonarQube AND 정적 분석'")


=== Match Query (AND) - 'SonarQube AND 정적 분석' ===
검색 결과: 0건



{'took': 1,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [ ]:
# [03-11] Match Query (50% 매칭)
# [목적] minimum_should_match로 검색어 중 일부만 매칭해도 결과에 포함합니다
# [개념] "50%" = 검색어 4개 중 2개 이상 매칭되면 결과에 포함
# Match Query with minimum_should_match
query = {
    "query": {
        "match": {
            "content": {
                "query": "연차 휴가 승인 정책",
                "minimum_should_match": "50%"  # 4개 토큰 중 2개 이상 매칭 시 결과 포함
            }
        }
    }
}

search_and_print(query, "Match Query (50% 매칭) - '연차 휴가 승인 정책'")


=== Match Query (50% 매칭) - '연차 휴가 승인 정책' ===
검색 결과: 1건

[4.2382] 휴가 및 근태 관리 규정
         카테고리: 인사



{'took': 2,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 1, 'relation': 'eq'},
  'max_score': 4.2381725,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '3',
    '_score': 4.2381725,
    '_source': {'title': '휴가 및 근태 관리 규정',
     'content': '연차휴가는 입사일 기준으로 부여되며, 1년 미만 근무자는 월 1일씩 발생합니다. 병가, 경조사 휴가, 출산휴가 등 특별휴가 제도가 있습니다. 근태 관리는 전자출결 시스템을 통해 이루어지며, 지각 및 조퇴 시 사전 승인이 필요합니다. 초과근무는 사전 승인 후 진행하며, 대체휴가 또는 수당으로 보상됩니다.',
     'category': '인사',
     'source': 'attendance_policy.pdf',
     'embedding': [-0.023854976519942284,
      0.0386076495051384,
      0.09063734114170074,
      0.03623565286397934,
      0.018146753311157227,
      0.0025190431624650955,
      -0.00531771220266819,
      0.02587985247373581,
      -0.009999033994972706,
      -1.8116921637556516e-05,
      0.0011112716747447848,
      0.010047245770692825,
      0.005356281064450741,
      0.004596952348947525,
      0.003169896313920617,
 

### Multi Match Query
여러 필드에서 동시에 검색

In [ ]:
# [03-12] Multi Match - 여러 필드 동시 검색
# [목적] title과 content 필드에서 동시에 검색하고, title에 3배 가중치를 줍니다
# [개념] title^3 = 제목에서 매칭되면 점수를 3배로 높임. 제목 매칭을 우선시하는 전략
# Multi Match Query
query = {
    "query": {
        "multi_match": {  # 여러 필드를 동시에 검색
            "query": "정책 안내",
            "fields": ["title^3", "content"],  # title 매칭 시 점수 3배 부스트
        }
    }
}

search_and_print(query, "Multi Match - title과 content에서 '정책 안내' 검색")


=== Multi Match - title과 content에서 '정책 안내' 검색 ===
검색 결과: 5건

[7.7731] 정보보안 정책
         카테고리: IT

[6.8167] 고객사 관리 정책
         카테고리: 영업

[5.1103] 사내 복지 제도 안내
         카테고리: 인사

[5.1103] 클라우드 서비스 요금제 안내
         카테고리: 제품

[5.1103] 사내 교육 프로그램 안내
         카테고리: 교육



{'took': 4,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 5, 'relation': 'eq'},
  'max_score': 7.773105,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '7',
    '_score': 7.773105,
    '_source': {'title': '정보보안 정책',
     'content': '모든 임직원은 정보보안 규정을 준수해야 합니다. 비밀번호는 8자 이상, 특수문자 포함이 필수이며, 90일마다 변경해야 합니다. 외부 USB 사용은 보안팀 승인 후 가능하며, 중요 문서는 DRM 적용이 필수입니다. 재택근무 시에는 VPN을 통해서만 사내 시스템에 접속해야 합니다.',
     'category': 'IT',
     'source': 'security_policy.pdf',
     'embedding': [0.012148148380219936,
      0.04270131513476372,
      0.004332282114773989,
      0.03681003674864769,
      0.031441036611795425,
      -0.014508836902678013,
      0.002637495519593358,
      0.03421954810619354,
      -0.015146014280617237,
      -0.04545893520116806,
      0.011688544414937496,
      0.041176266968250275,
      0.03503429889678955,
      0.02953995019197464,
      -0.011009585112333298,
      -0.024902136996388435,


### Match Phrase Query
정확한 구문(어순 유지) 검색

In [ ]:
# [03-13] Match Phrase - 구문 검색
# [목적] "연차 휴가"가 정확히 연속으로 나오는 문서만 검색합니다
# [개념] Match Phrase = 단어 순서까지 일치해야 매칭. "휴가 연차"는 매칭 안 됨
# Match Phrase Query
query = {
    "query": {
        "match_phrase": {  # 단어 순서까지 정확히 일치해야 매칭
            "content": "연차 휴가"  # "연차" 바로 뒤에 "휴가"가 나와야 함
        }
    }
}

search_and_print(query, "Match Phrase - '연차 휴가' (연속 구문)")


=== Match Phrase - '연차 휴가' (연속 구문) ===
검색 결과: 0건



{'took': 2,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

## 3.4 Exact Query (Term)
### Term Query
- 검색어를 **분석하지 않고** 그대로 검색
- keyword 필드에 사용
- 정확한 값 매칭 (필터, ID 검색 등)

In [ ]:
# [03-14] Term Query - 정확한 값 매칭
# [목적] category 필드에서 정확히 "인사"인 문서만 필터링합니다
# [개념] Term = 토큰화 없이 원본 그대로 매칭. keyword 타입 필드에 사용합니다
# Term Query - keyword 필드
query = {
    "query": {
        "term": {  # Term: 토큰화 없이 원본 그대로 정확 매칭
            "category": "인사"  # keyword 타입 필드에서 정확히 "인사"인 문서
        }
    }
}

search_and_print(query, "Term Query - category가 '인사'인 문서")


=== Term Query - category가 '인사'인 문서 ===
검색 결과: 3건

[1.7918] 사내 복지 제도 안내
         카테고리: 인사

[1.7918] 신입사원 온보딩 프로그램
         카테고리: 인사

[1.7918] 휴가 및 근태 관리 규정
         카테고리: 인사



{'took': 3,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 3, 'relation': 'eq'},
  'max_score': 1.7917595,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '1',
    '_score': 1.7917595,
    '_source': {'title': '사내 복지 제도 안내',
     'content': '당사는 직원들의 워라밸을 위해 다양한 복지 제도를 운영하고 있습니다. 유연근무제를 통해 출퇴근 시간을 자유롭게 조정할 수 있으며, 주 2회 재택근무가 가능합니다. 연차 사용을 적극 장려하고 있으며, 미사용 연차에 대한 보상도 제공합니다. 또한 건강검진, 단체보험, 자기개발비 지원 등의 혜택이 있습니다.',
     'category': '인사',
     'source': 'hr_policy.pdf',
     'embedding': [-0.02303275093436241,
      0.05416445806622505,
      0.05437871813774109,
      0.043237295001745224,
      -0.0027237567119300365,
      0.004753852728754282,
      -0.018436912447214127,
      0.029846159741282463,
      -0.018501190468668938,
      -0.03888785466551781,
      0.018918994814157486,
      -0.026439454406499863,
      0.005361809860914946,
      -0.006599150598049164,
      0.034795522689819336,
      -0.

In [ ]:
# [03-15] Terms Query - 여러 값 중 하나
# [목적] category가 "인사" 또는 "재무"인 문서를 필터링합니다
# [개념] Terms = OR 조건의 정확 매칭. SQL의 IN 절과 유사합니다
# Terms Query - 여러 값 중 하나
query = {
    "query": {
        "terms": {  # Terms: 여러 값 중 하나라도 매칭 (SQL의 IN 절)
            "category": ["인사", "재무"]  # "인사" OR "재무" 매칭
        }
    }
}

search_and_print(query, "Terms Query - category가 '인사' 또는 '재무'")


=== Terms Query - category가 '인사' 또는 '재무' ===
검색 결과: 5건

[1.0000] 사내 복지 제도 안내
         카테고리: 인사

[1.0000] 신입사원 온보딩 프로그램
         카테고리: 인사

[1.0000] 휴가 및 근태 관리 규정
         카테고리: 인사

[1.0000] 예산 수립 및 집행 가이드
         카테고리: 재무

[1.0000] 경비 처리 규정
         카테고리: 재무



{'took': 4,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 5, 'relation': 'eq'},
  'max_score': 1.0,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '1',
    '_score': 1.0,
    '_source': {'title': '사내 복지 제도 안내',
     'content': '당사는 직원들의 워라밸을 위해 다양한 복지 제도를 운영하고 있습니다. 유연근무제를 통해 출퇴근 시간을 자유롭게 조정할 수 있으며, 주 2회 재택근무가 가능합니다. 연차 사용을 적극 장려하고 있으며, 미사용 연차에 대한 보상도 제공합니다. 또한 건강검진, 단체보험, 자기개발비 지원 등의 혜택이 있습니다.',
     'category': '인사',
     'source': 'hr_policy.pdf',
     'embedding': [-0.02303275093436241,
      0.05416445806622505,
      0.05437871813774109,
      0.043237295001745224,
      -0.0027237567119300365,
      0.004753852728754282,
      -0.018436912447214127,
      0.029846159741282463,
      -0.018501190468668938,
      -0.03888785466551781,
      0.018918994814157486,
      -0.026439454406499863,
      0.005361809860914946,
      -0.006599150598049164,
      0.034795522689819336,
      -0.020150978118

### Match vs Term 비교

| 구분 | Match | Term |
|------|-------|------|
| 분석 | O (토큰화) | X |
| 용도 | 전문 검색 | 정확한 매칭 |
| 필드 타입 | text | keyword |
| 예시 | 문서 내용 검색 | 카테고리 필터 |

In [ ]:
# [03-16] text 필드에 Term 쿼리 사용 시 문제
# [목적] text 필드에 Term 쿼리를 사용하면 결과가 나오지 않는 문제를 시연합니다
# [개념] text 필드는 저장 시 토큰화됨 → Term은 토큰화 안 함 → 전체 문장 매칭 불가
# [주의] 이것은 의도된 실패 예시입니다. text에는 Match, keyword에는 Term을 사용하세요
# 주의: text 필드에 term 쿼리 사용 시 문제
# text 필드는 토큰화되어 저장되므로 정확한 문장 매칭 불가

# 잘못된 예: text 필드에 term 사용
query = {
    "query": {
        "term": {
            "title": "회사 연차 휴가 정책"  # text 필드는 토큰화 저장 → 전체 문장과 일치하는 토큰 없음 → 결과 0건
        }
    }
}

response = search_and_print(query, "[주의] text 필드에 term 쿼리 - 결과 없음")


=== [주의] text 필드에 term 쿼리 - 결과 없음 ===
검색 결과: 0건



## 3.5 복합 쿼리 (Bool Query)
### Bool Query 구조

| 절 | 설명 | 점수 영향 |
|---|------|----------|
| must | AND - 반드시 매칭 | O |
| should | OR - 매칭되면 가점 | O |
| must_not | NOT - 제외 | X |
| filter | AND - 반드시 매칭 | X (캐싱) |

In [ ]:
# [03-17] Bool Query - must + filter 조합
# [목적] content에 "정책" 포함(must) AND category가 "인사"(filter)인 문서를 검색합니다
# [개념] must = 점수에 반영되는 필수 조건, filter = 점수 무관 필터 (더 빠르고 캐싱됨)
# Bool Query 기본
query = {
    "query": {
        "bool": {  # Bool: 여러 쿼리를 논리 조합
            "must": [  # must: 반드시 매칭 + 점수 계산에 반영
                {"match": {"content": "정책"}}
            ],
            "filter": [  # filter: 반드시 매칭 + 점수 계산 없음 (캐싱 가능, 더 빠름)
                {"term": {"category": "인사"}}  # 정확히 "인사" 카테고리만
            ]
        }
    }
}

search_and_print(query, "Bool Query - content에 '정책' AND category='인사'")


=== Bool Query - content에 '정책' AND category='인사' ===
검색 결과: 0건



{'took': 1,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [ ]:
# [03-18] Bool Query - must_not 제외 조건
# [목적] "정책" 포함하되 category가 "재무"인 문서는 제외합니다
# [개념] must_not = 해당 조건에 매칭되는 문서를 결과에서 제거
# Bool Query - must_not 사용
query = {
    "query": {
        "bool": {
            "must": [
                {"match": {"content": "정책"}}  # "정책" 포함 필수
            ],
            "must_not": [  # must_not: 매칭되는 문서를 결과에서 제외
                {"term": {"category": "재무"}}  # "재무" 카테고리 제외
            ]
        }
    }
}

search_and_print(query, "Bool Query - '정책' 포함 BUT category≠'재무'")


=== Bool Query - '정책' 포함 BUT category≠'재무' ===
검색 결과: 0건



{'took': 1,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [ ]:
# [03-19] Bool Query - should 가점
# [목적] "정책" 필수, "휴가"/"연차" 포함 시 가점을 부여합니다
# [개념] should = 매칭되면 점수 상승(가점). 필수는 아니지만 관련성 높은 문서를 상위로 올림
# Bool Query - should로 가점
query = {
    "query": {
        "bool": {
            "must": [
                {"match": {"content": "정책"}}  # 필수 조건
            ],
            "should": [  # should: 매칭 시 점수 가산 (필수 아님)
                {"match": {"title": "휴가"}},  # 제목에 '휴가' 있으면 점수 상승
                {"match": {"content": "연차"}}  # 내용에 '연차' 있으면 점수 상승
            ]
        }
    }
}

search_and_print(query, "Bool Query - '정책' 필수, '휴가'/'연차' 가점")


=== Bool Query - '정책' 필수, '휴가'/'연차' 가점 ===
검색 결과: 0건



{'took': 1,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 0, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

### Filter vs Must

```
filter:
- 점수 계산 안 함 → 빠름
- 결과 캐싱 → 재사용 시 더 빠름
- 용도: 정확한 조건 (카테고리, 날짜 범위 등)

must:
- 점수 계산 함 → 상대적 느림
- 용도: 관련성 순위가 필요한 검색
```

In [ ]:
# [03-20] 실무 패턴 - 키워드 검색 + 필터
# [목적] 실무에서 자주 쓰는 패턴: multi_match 검색 + 카테고리 필터 조합입니다
# [개념] must(검색) + filter(필터)는 가장 일반적인 Bool Query 패턴입니다
# 실무 패턴: 키워드 검색 + 필터
query = {
    "query": {
        "bool": {
            "must": [  # 검색 조건: 점수 계산에 반영
                {
                    "multi_match": {  # 여러 필드에서 동시 검색
                        "query": "교육 프로그램",
                        "fields": ["title^2", "content"]  # title에 2배 가중치
                    }
                }
            ],
            "filter": [  # 필터 조건: 점수 무관, 캐싱 가능
                {"term": {"category": "인사"}}  # "인사" 카테고리만 필터링
            ]
        }
    }
}

search_and_print(query, "실무 패턴 - 키워드 검색 + 카테고리 필터")


=== 실무 패턴 - 키워드 검색 + 카테고리 필터 ===
검색 결과: 1건

[4.5445] 신입사원 온보딩 프로그램
         카테고리: 인사



{'took': 3,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 1, 'relation': 'eq'},
  'max_score': 4.5444946,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '2',
    '_score': 4.5444946,
    '_source': {'title': '신입사원 온보딩 프로그램',
     'content': '신입사원을 위한 체계적인 온보딩 프로그램을 운영합니다. 입사 첫 주에는 회사 소개, 조직 문화, 업무 시스템 교육이 진행됩니다. 멘토링 제도를 통해 선배 사원이 1:1로 업무 적응을 도와드립니다. 3개월간의 수습 기간 동안 정기적인 피드백 세션이 진행되며, 부서별 실무 교육도 병행됩니다.',
     'category': '인사',
     'source': 'onboarding_guide.pdf',
     'embedding': [-0.016509652137756348,
      0.017363054677844048,
      0.022420255467295647,
      0.026255300268530846,
      0.0022033064160495996,
      0.008728939108550549,
      0.023031333461403847,
      0.003995715174823999,
      -0.035442546010017395,
      -0.017700202763080597,
      0.040752608329057693,
      0.003015882568433881,
      0.012390141375362873,
      -0.006100248079746962,
      0.05402775853872299,
      0.

## 3.6 Range Query (범위 검색)

In [ ]:
# [03-21] Range Query - 날짜 범위 검색
# [목적] 특정 기간(2024년) 내 생성된 문서만 검색합니다
# [개념] Range = 숫자/날짜의 범위 조건. gte(이상), lte(이하)
# Range Query - 날짜 범위
query = {
    "query": {
        "range": {  # Range: 숫자/날짜의 범위 조건 검색
            "created_at": {
                "gte": "2024-01-01",  # gte: 이상 (greater than or equal)
                "lte": "2024-12-31"   # lte: 이하 (less than or equal)
            }
        }
    }
}

search_and_print(query, "Range Query - 2024년 문서")


=== Range Query - 2024년 문서 ===
검색 결과: 20건

[1.0000] 사내 복지 제도 안내
         카테고리: 인사

[1.0000] API 연동 가이드
         카테고리: 제품

[1.0000] 신입사원 온보딩 프로그램
         카테고리: 인사

[1.0000] 시스템 장애 대응 매뉴얼
         카테고리: IT

[1.0000] 클라우드 서비스 요금제 안내
         카테고리: 제품

[1.0000] 고객사 관리 정책
         카테고리: 영업

[1.0000] 휴가 및 근태 관리 규정
         카테고리: 인사

[1.0000] 영업 프로세스 가이드
         카테고리: 영업

[1.0000] 개발 환경 설정 가이드
         카테고리: IT

[1.0000] 정보보안 정책
         카테고리: IT



{'took': 6,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 20, 'relation': 'eq'},
  'max_score': 1.0,
  'hits': [{'_index': 'student_00_company_docs',
    '_id': '1',
    '_score': 1.0,
    '_source': {'title': '사내 복지 제도 안내',
     'content': '당사는 직원들의 워라밸을 위해 다양한 복지 제도를 운영하고 있습니다. 유연근무제를 통해 출퇴근 시간을 자유롭게 조정할 수 있으며, 주 2회 재택근무가 가능합니다. 연차 사용을 적극 장려하고 있으며, 미사용 연차에 대한 보상도 제공합니다. 또한 건강검진, 단체보험, 자기개발비 지원 등의 혜택이 있습니다.',
     'category': '인사',
     'source': 'hr_policy.pdf',
     'embedding': [-0.02303275093436241,
      0.05416445806622505,
      0.05437871813774109,
      0.043237295001745224,
      -0.0027237567119300365,
      0.004753852728754282,
      -0.018436912447214127,
      0.029846159741282463,
      -0.018501190468668938,
      -0.03888785466551781,
      0.018918994814157486,
      -0.026439454406499863,
      0.005361809860914946,
      -0.006599150598049164,
      0.034795522689819336,
      -0.02015097811

## 3.7 실습: 복합 검색 시나리오

In [ ]:
# [03-22] keyword_search 함수 정의
# [목적] 키워드 검색 + 카테고리 필터를 하나로 묶은 재사용 가능한 함수입니다
# [개념] multi_match + Bool Query + filter를 조합한 실무형 검색 함수
def keyword_search(
    query_text: str,
    category: str = None,
    size: int = 5
) -> list[dict]:
    """키워드 검색 함수"""

    # 기본 쿼리: multi_match
    must_clauses = [
        {
            "multi_match": {
                "query": query_text,
                "fields": ["title^2", "content"],  # title에 2배 가중치 부스트
            }
        }
    ]

    # 필터 조건 (카테고리가 지정된 경우만 추가)
    filter_clauses = []
    if category:
        filter_clauses.append({"term": {"category": category}})  # 카테고리 정확 매칭 필터

    # Bool Query 구성: must(검색) + filter(필터)
    search_body = {
        "query": {
            "bool": {
                "must": must_clauses,  # 키워드 검색 조건
                "filter": filter_clauses if filter_clauses else None,  # 선택적 필터
            }
        },
        "size": size,  # 반환할 최대 문서 수
    }

    # None 값 제거 (filter가 비어있으면 키 자체를 삭제)
    if not filter_clauses:
        del search_body["query"]["bool"]["filter"]

    response = client.search(index=INDEX_NAME, body=search_body)  # 검색 실행

    # 결과 파싱: 필요한 필드만 추출
    results = []
    for hit in response['hits']['hits']:
        results.append({
            "id": hit['_id'],  # 문서 고유 ID
            "score": hit['_score'],  # BM25 관련성 점수
            "title": hit['_source']['title'],
            "category": hit['_source']['category'],
            "content": hit['_source']['content'][:100] + "...",  # 내용은 100자까지만
        })

    return results


In [ ]:
# [03-23] 키워드 검색 강점 테스트
# [목적] 정확한 기술 용어(Kubernetes, CI/CD)에서 키워드 검색의 강점을 확인합니다
# [개념] 키워드 검색은 정확한 용어가 문서에 포함되어 있을 때 가장 잘 동작합니다
# 검색 테스트 - 키워드 검색의 강점
print("=== 키워드 검색의 강점 (정확한 기술 용어) ===\n")

print("검색: 'Kubernetes'")
results = keyword_search("Kubernetes")
for r in results:
    print(f"  [{r['score']:.2f}] {r['title']} ({r['category']})")

print("\n검색: 'CI/CD 파이프라인'")
results = keyword_search("CI/CD 파이프라인")
for r in results:
    print(f"  [{r['score']:.2f}] {r['title']} ({r['category']})")

=== 키워드 검색의 강점 (정확한 기술 용어) ===

검색: 'Kubernetes'
  [2.73] 머신러닝 모델 배포 가이드 (AI/ML)

검색: 'CI/CD 파이프라인'
  [5.40] 개발 환경 설정 가이드 (IT)
  [2.64] 영업 프로세스 가이드 (영업)


## 3.8 키워드 검색의 한계
### 동의어 문제

In [ ]:
# [03-24] 키워드 검색 한계 - 동의어 문제
# [목적] "진급"→"승진", "쉬는 날"→"휴가" 등 동의어를 인식하지 못하는 한계를 시연합니다
# [개념] 키워드 검색은 정확한 단어 매칭만 수행 → 동의어/유사어 처리 불가
# 키워드 검색의 한계 1: 동의어 문제
print("=== 키워드 검색의 한계: 동의어 ===\n")

# "진급"으로 검색 → "승진" 문서 검색 안됨
print("검색: '진급' (동의어: 승진)")
results = keyword_search("진급")
if results:
    for r in results:
        print(f"  {r['title']}")
else:
    print("  결과 없음 - '승진' 문서를 찾지 못함")

print()

# "쉬는 날"로 검색 → "휴가" 문서 검색 안됨
print("검색: '쉬는 날' (동의어: 휴가)")
results = keyword_search("쉬는 날")
if results:
    for r in results:
        print(f"  {r['title']}")
else:
    print("  결과 없음 - '휴가' 문서를 찾지 못함")

=== 키워드 검색의 한계: 동의어 ===

검색: '진급' (동의어: 승진)
  결과 없음 - '승진' 문서를 찾지 못함

검색: '쉬는 날' (동의어: 휴가)
  결과 없음 - '휴가' 문서를 찾지 못함


### 의미 검색 불가

In [ ]:
# [03-25] 키워드 검색 한계 - 자연어 질문
# [목적] 자연어 질문("며칠 쉴 수 있나요?")에 대해 키워드 검색이 실패하는 것을 확인합니다
# [개념] 일상 표현과 문서 용어가 다르면 키워드 검색은 연결하지 못합니다
# [주의] 이 한계를 해결하는 것이 4장의 시맨틱 검색입니다
# 키워드 검색의 한계 2: 자연어 질문
print("=== 키워드 검색의 한계: 자연어 질문 ===\n")

test_questions = [
    "며칠 쉴 수 있나요?",
    "집에서 일하고 싶어요",
    "어떻게 하면 올라갈 수 있나요?",  # 승진
]

for q in test_questions:
    print(f"질문: '{q}'")
    results = keyword_search(q)
    if results:
        for r in results[:2]:
            print(f"  → {r['title']}")
    else:
        print("  → 결과 없음")
    print()

=== 키워드 검색의 한계: 자연어 질문 ===

질문: '며칠 쉴 수 있나요?'
  → API 연동 가이드
  → 사내 복지 제도 안내

질문: '집에서 일하고 싶어요'
  → 결과 없음

질문: '어떻게 하면 올라갈 수 있나요?'
  → API 연동 가이드
  → 사내 복지 제도 안내



## 3.9 정리
### 학습 내용

| 쿼리 | 용도 | 특징 |
|------|------|------|
| Match | 전문 검색 | 토큰화 후 OR/AND 매칭 |
| Multi Match | 다중 필드 검색 | 필드별 가중치 |
| Match Phrase | 구문 검색 | 어순 유지 |
| Term | 정확 매칭 | keyword 필드용 |
| Bool | 복합 쿼리 | must, should, must_not, filter |
| Range | 범위 검색 | 숫자, 날짜 |

### 키워드 검색 한계
- 동의어 처리 불가
- 의미 기반 검색 불가
- → 벡터 검색으로 보완

### 다음 학습
- 시맨틱 검색 (벡터 검색)
- 하이브리드 검색